# Détection de Vulnérabilités en Cryptographie Post-Quantique

## Analyse Exploratoire et Expérimentations Machine Learning

**Auteurs :** Projet académique de cryptographie post-quantique  
**Date :** 2026  
**Contexte :** ce notebook présente l'exploration du dataset, l'analyse des features et les expérimentations Machine Learning pour détecter des vulnérabilités dans des implémentations McEliece, HQC et Classic McEliece 6688128.

## SECTION 2 — Imports

In [ ]:
# Ajout des chemins du projet pour permettre les imports depuis src/ si besoin.
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../src')

# Imports principaux pour la manipulation de données, la visualisation et le ML.
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, cross_val_score, learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Configuration graphique globale.
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

# Chemins robustes : le notebook peut être lancé depuis notebooks/ ou depuis la racine.
PROJECT_ROOT = Path('..') if Path('../data').exists() else Path('.')
DATASET_PATH = PROJECT_ROOT / 'data' / 'dataset.csv'
FEATURE_MATRIX_PATH = PROJECT_ROOT / 'data' / 'feature_matrix.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'model.pkl'
FEATURE_COLS_PATH = PROJECT_ROOT / 'models' / 'feature_cols.pkl'

print(f'Version pandas : {pd.__version__}')
print(f'Version scikit-learn : {sklearn.__version__}')
print(f'Racine projet : {PROJECT_ROOT.resolve()}')

## SECTION 3 — Dataset Loading

In [ ]:
# Chargement du dataset annoté et de la matrice de features.
try:
    dataset_df = pd.read_csv(DATASET_PATH)
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)

    # Affichage des dimensions et des types de colonnes.
    print('Shape dataset.csv :', dataset_df.shape)
    print('Shape feature_matrix.csv :', feature_df.shape)
    print('\nTypes dataset.csv :')
    print(dataset_df.dtypes)
    print('\nTypes feature_matrix.csv :')
    print(feature_df.dtypes)

    # Aperçu des premières lignes.
    display(feature_df.head())

    # Distribution des classes 0/1.
    print('\nDistribution des classes :')
    print(feature_df['label'].value_counts().sort_index())
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')
    dataset_df = pd.DataFrame()
    feature_df = pd.DataFrame()

## SECTION 4 — Dataset Visualization

In [ ]:
# Visualisation globale du dataset : labels, types, schémas et sévérités.
try:
    if dataset_df.empty or feature_df.empty:
        dataset_df = pd.read_csv(DATASET_PATH)
        feature_df = pd.read_csv(FEATURE_MATRIX_PATH)

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    secure_color = '#1b2a4a'
    vulnerable_color = '#8b1a1a'

    # 1. Distribution des labels.
    label_counts = feature_df['label'].astype(int).value_counts().reindex([0, 1], fill_value=0)
    axes[0, 0].bar(['Sécurisé', 'Vulnérable'], label_counts.values, color=[secure_color, vulnerable_color], label='Fichiers')
    axes[0, 0].set_title('Distribution des labels')
    axes[0, 0].set_xlabel('Classe')
    axes[0, 0].set_ylabel('Nombre de fichiers')
    axes[0, 0].legend()

    # 2. Distribution des types de vulnérabilités.
    vuln_counts = dataset_df.get('vulnerability_type', pd.Series(dtype=str)).replace('', 'secure').fillna('secure').value_counts()
    axes[0, 1].bar(vuln_counts.index.astype(str), vuln_counts.values, color=vulnerable_color, label='Types')
    axes[0, 1].set_title('Distribution des types de vulnérabilités')
    axes[0, 1].set_xlabel('Type')
    axes[0, 1].set_ylabel('Nombre de fichiers')
    axes[0, 1].tick_params(axis='x', rotation=35)
    axes[0, 1].legend()

    # 3. Distribution par schéma cryptographique.
    file_paths = dataset_df.get('file_path', pd.Series(dtype=str)).astype(str).str.lower()
    schemes = np.select(
        [file_paths.str.contains('hqc'), file_paths.str.contains('python'), file_paths.str.contains('mceliece')],
        ['hqc', 'python', 'mceliece'],
        default='autre'
    )
    scheme_counts = pd.Series(schemes).value_counts().reindex(['mceliece', 'hqc', 'python', 'autre']).dropna()
    axes[1, 0].bar(scheme_counts.index, scheme_counts.values, color=secure_color, label='Schémas')
    axes[1, 0].set_title('Distribution par schéma cryptographique')
    axes[1, 0].set_xlabel('Schéma')
    axes[1, 0].set_ylabel('Nombre de fichiers')
    axes[1, 0].legend()

    # 4. Distribution par niveau de sévérité approximé à partir du type.
    severity_map = {'weak_prng': 'Critique', 'timing_leak': 'Critique', 'memory_leak': 'Élevée', 'weak_params': 'Élevée', 'secure': 'Faible'}
    severity = dataset_df.get('vulnerability_type', pd.Series(dtype=str)).replace('', 'secure').fillna('secure').map(severity_map).fillna('Faible')
    severity_counts = severity.value_counts().reindex(['Faible', 'Élevée', 'Critique'], fill_value=0)
    axes[1, 1].bar(severity_counts.index, severity_counts.values, color=[secure_color, '#c47f2c', vulnerable_color], label='Sévérité')
    axes[1, 1].set_title('Distribution par niveau de sévérité')
    axes[1, 1].set_xlabel('Sévérité')
    axes[1, 1].set_ylabel('Nombre de fichiers')
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 5 — Feature Analysis

In [ ]:
# Analyse statistique et visuelle des features utilisées par le modèle.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']

    # Statistiques descriptives principales.
    stats = feature_df[feature_cols].agg(['mean', 'std', 'min', 'max']).T
    display(stats)

    # Heatmap de corrélation entre features.
    plt.figure(figsize=(9, 7))
    sns.heatmap(feature_df[feature_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True)
    plt.title('Corrélation entre les features')
    plt.xlabel('Features')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()

    # Distribution du nombre de lignes selon la classe.
    plt.figure(figsize=(10, 5))
    secure_loc = feature_df.loc[feature_df['label'].astype(int) == 0, 'loc']
    vulnerable_loc = feature_df.loc[feature_df['label'].astype(int) == 1, 'loc']
    plt.hist(secure_loc, bins=30, alpha=0.65, color='#1b2a4a', label='Sécurisé')
    plt.hist(vulnerable_loc, bins=30, alpha=0.65, color='#8b1a1a', label='Vulnérable')
    plt.title('Distribution du nombre de lignes par classe')
    plt.xlabel('Nombre de lignes de code')
    plt.ylabel('Nombre de fichiers')
    plt.legend(title='Classe')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 6 — Feature Importance

In [ ]:
# Chargement du modèle sauvegardé et visualisation de l'importance des features.
try:
    model = joblib.load(MODEL_PATH)
    feature_cols = joblib.load(FEATURE_COLS_PATH)

    # Construction du tableau d'importance.
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=True)

    top_features = importance_df.tail(3)['feature'].tolist()
    colors = ['#8b1a1a' if feature in top_features else '#1b2a4a' for feature in importance_df['feature']]

    # Graphique horizontal avec labels numériques.
    plt.figure(figsize=(10, 6))
    bars = plt.barh(importance_df['feature'], importance_df['importance'], color=colors, label='Importance')
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.005, bar.get_y() + bar.get_height() / 2, f'{width:.3f}', va='center')
    plt.title('Importance des features du Random Forest')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.legend(title='Top 3 en rouge')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 7 — Model Training Comparison

In [ ]:
# Comparaison de plusieurs modèles supervisés avec validation croisée.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']
    X = feature_df[feature_cols]
    y = feature_df['label'].astype(int)

    # Définition des modèles à comparer.
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
        'Logistic Regression': Pipeline([
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
        ]),
        'Support Vector Machine': Pipeline([
            ('scaler', StandardScaler()),
            ('model', SVC(class_weight='balanced', probability=True, random_state=42))
        ]),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    }

    # Calcul des métriques avec cross_val_score.
    rows = []
    for name, clf in models.items():
        rows.append({
            'Modèle': name,
            'F1-score': cross_val_score(clf, X, y, cv=5, scoring='f1').mean(),
            'Precision': cross_val_score(clf, X, y, cv=5, scoring='precision').mean(),
            'Recall': cross_val_score(clf, X, y, cv=5, scoring='recall').mean(),
            'AUC-ROC': cross_val_score(clf, X, y, cv=5, scoring='roc_auc').mean(),
        })

    comparison_df = pd.DataFrame(rows).sort_values('F1-score', ascending=False)
    display(comparison_df)

    # Graphique groupé des performances.
    plot_df = comparison_df.set_index('Modèle')
    ax = plot_df.plot(kind='bar', figsize=(12, 6), color=['#1b2a4a', '#8b1a1a', '#2d6a2d', '#c47f2c'])
    ax.set_title('Comparaison des modèles par métrique')
    ax.set_xlabel('Modèle')
    ax.set_ylabel('Score moyen en validation croisée')
    ax.legend(title='Métrique')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 8 — Confusion Matrix

In [ ]:
# Entraînement d'un Random Forest et affichage de la matrice de confusion.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']
    X = feature_df[feature_cols]
    y = feature_df['label'].astype(int)

    # Split 80/20 stratifié.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    # Calcul des comptes et des pourcentages.
    cm = confusion_matrix(y_test, y_pred)
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    labels = np.array([[f'{cm[i, j]}\n{cm_pct[i, j]:.1f}%' for j in range(cm.shape[1])] for i in range(cm.shape[0])])

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', xticklabels=['Secure', 'Vulnerable'], yticklabels=['Secure', 'Vulnerable'])
    plt.title('Matrice de confusion du Random Forest')
    plt.xlabel('Classe prédite')
    plt.ylabel('Classe réelle')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 9 — ROC Curve

In [ ]:
# Courbe ROC du Random Forest sur le split de test.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']
    X = feature_df[feature_cols]
    y = feature_df['label'].astype(int)

    # Entraînement puis prédiction des probabilités.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_train, y_train)
    y_score = rf.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color='#1b2a4a', linewidth=2, label=f'Random Forest (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Référence aléatoire')
    plt.title('Courbe ROC du Random Forest')
    plt.xlabel('Taux de faux positifs')
    plt.ylabel('Taux de vrais positifs')
    plt.legend(title='Modèle')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 10 — Learning Curve

In [ ]:
# Courbe d'apprentissage pour observer la stabilité du Random Forest.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']
    X = feature_df[feature_cols]
    y = feature_df['label'].astype(int)

    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    train_sizes, train_scores, cv_scores = learning_curve(
        rf, X, y, cv=5, scoring='f1', train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1
    )

    # Moyennes et écarts-types pour les zones ombrées.
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    cv_mean = cv_scores.mean(axis=1)
    cv_std = cv_scores.std(axis=1)

    plt.figure(figsize=(9, 6))
    plt.plot(train_sizes, train_mean, marker='o', color='#1b2a4a', label='Score entraînement')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='#1b2a4a')
    plt.plot(train_sizes, cv_mean, marker='o', color='#8b1a1a', label='Score validation croisée')
    plt.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std, alpha=0.2, color='#8b1a1a')
    plt.title('Courbe d’apprentissage du Random Forest')
    plt.xlabel('Taille du jeu d’entraînement')
    plt.ylabel('F1-score')
    plt.legend(title='Type de score')
    plt.tight_layout()
    plt.show()
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 11 — Error Analysis

In [ ]:
# Analyse des erreurs : vrais positifs, faux négatifs et faux positifs.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']
    X = feature_df[feature_cols]
    y = feature_df['label'].astype(int)

    # Split avec conservation des index pour retrouver les chemins de fichiers.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_train, y_train)
    y_pred = pd.Series(rf.predict(X_test), index=X_test.index)
    y_true = pd.Series(y_test, index=X_test.index)

    # Préparation des catégories d'analyse.
    analysis_df = feature_df.loc[X_test.index, ['file_path'] + feature_cols].copy()
    analysis_df['label_reel'] = y_true
    analysis_df['prediction'] = y_pred

    true_positives = analysis_df[(analysis_df['label_reel'] == 1) & (analysis_df['prediction'] == 1)].head(5)
    false_negatives = analysis_df[(analysis_df['label_reel'] == 1) & (analysis_df['prediction'] == 0)].head(5)
    false_positives = analysis_df[(analysis_df['label_reel'] == 0) & (analysis_df['prediction'] == 1)].head(5)

    print('Vrais positifs : vulnérabilités correctement détectées')
    display(true_positives)
    print('Faux négatifs : vulnérabilités manquées')
    display(false_negatives)
    print('Faux positifs : fausses alertes')
    display(false_positives)
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 12 — Hyperparameter Optimization

In [ ]:
# Optimisation des hyperparamètres du Random Forest par GridSearchCV.
try:
    feature_df = pd.read_csv(FEATURE_MATRIX_PATH)
    feature_cols = ['has_weak_prng', 'has_timing_leak', 'has_memory_leak', 'has_weak_params', 'dangerous_func_count', 'loc', 'comment_lines']
    X = feature_df[feature_cols]
    y = feature_df['label'].astype(int)

    # Modèle par défaut et grille demandée.
    default_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }

    grid = GridSearchCV(default_model, param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid.fit(X, y)

    # Comparaison du modèle optimisé avec le modèle par défaut.
    default_score = cross_val_score(default_model, X, y, cv=5, scoring='f1').mean()
    best_score = grid.best_score_

    print('Meilleurs paramètres :', grid.best_params_)
    print(f'Meilleur F1-score : {best_score:.4f}')
    print(f'F1-score modèle par défaut : {default_score:.4f}')

    comparison = pd.DataFrame({
        'Modèle': ['Random Forest par défaut', 'Random Forest optimisé'],
        'F1-score': [default_score, best_score]
    })
    display(comparison)
except FileNotFoundError as exc:
    print(f'Fichier introuvable : {exc}')

## SECTION 13 — Final Results Summary

| Metric | Value |
|--------|-------|
| F1-Score (cross-val) | 0.9255 |
| AUC-ROC | 0.8582 |
| Accuracy | 0.87 |
| Training samples | 585 |
| Vulnerability types | 4 |
| Cryptographic schemes | 3 |

## SECTION 14 — Conclusions

Les résultats montrent que l'association de règles statiques et d'un modèle Random Forest fonctionne bien pour repérer les vulnérabilités explicites présentes dans le dataset. Les features binaires comme `has_weak_prng`, `has_timing_leak`, `has_memory_leak` et `has_weak_params` sont particulièrement utiles, car elles capturent directement des patterns de sécurité critiques.

L'approche reste toutefois limitée par la couverture du dataset et par la nature des règles utilisées. Les vulnérabilités plus subtiles, dépendantes du contexte cryptographique ou de chemins d'exécution complexes, peuvent ne pas être détectées. Le modèle apprend aussi les patterns présents dans les fichiers annotés, ce qui peut réduire sa capacité de généralisation vers des implémentations très différentes.

Les améliorations futures incluent l'ajout du schéma BIKE, l'intégration de nouveaux schémas post-quantiques, l'utilisation de modèles de deep learning sur le code source, et l'enrichissement des features avec des graphes de flot de données ou des représentations AST plus détaillées.

Dans le cadre académique, ce projet constitue une base expérimentale pour étudier la détection automatique de vulnérabilités dans la cryptographie post-quantique et comparer des approches statiques, statistiques et hybrides.